# Training and classification with Dismir

In [1]:
import sys
sys.path.append("X:/KULeuven-Masters/Master Thesis/methyldl/methyldl")

In [2]:
import sys
print(sys.executable)


/home/luna.kuleuven.be/u0169940/.cache/pypoetry/virtualenvs/methyldl-GStZGe-R-py3.12/bin/python


In [3]:
import pandas as pd
import os
from methyldl.modelling.dismir import Dismir
from methyldl.data.dataset import extract_optimal_subsequence_dataframe_chunked
from tqdm import tqdm

/home/luna.kuleuven.be/u0169940/.cache/pypoetry/virtualenvs/methyldl-GStZGe-R-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# data_path = '/home/luna.kuleuven.be/u0169940/Data/Loyfer/TrainingDataWithRejection/'
data_path = '/home/luna.kuleuven.be/u0169940/Data/Loyfer/TrainingDataWithRejection_hg38_mincpg_4_minlen_10/'

In [5]:
import pandas as pd
import numpy as np
import ast
from typing import List, Tuple, Any

def extract_overlapping_regions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract overlapping regions with DMRs from DNA methylation reads dataset.
    
    Parameters:
    df (pd.DataFrame): Input dataset with methylation reads
    
    Returns:
    pd.DataFrame: New dataset with only overlapping regions
    """
    
    # List to store the new records
    new_records = []
    
    for idx, row in df.iterrows():
        # Handle overlap_locations (numpy array of arrays)
        overlap_locations = row['overlap_locations']
        if overlap_locations is None or (hasattr(overlap_locations, '__len__') and len(overlap_locations) == 0):
            continue
        
        # Convert numpy array to list if needed
        if isinstance(overlap_locations, np.ndarray):
            overlap_locations = overlap_locations.tolist()
        elif isinstance(overlap_locations, str):
            try:
                overlap_locations = ast.literal_eval(overlap_locations)
            except (ValueError, SyntaxError):
                continue
        
        # Handle areastats (numpy array)
        areastats = row['areastats']
        if isinstance(areastats, np.ndarray):
            areastats = areastats.tolist()
        elif isinstance(areastats, str):
            try:
                areastats = ast.literal_eval(areastats)
            except (ValueError, SyntaxError):
                areastats = []
        elif areastats is None:
            areastats = []
        
        # Handle overlapping_with_dmrs (numpy array)
        overlapping_dmrs = row['dmr_types']
        if isinstance(overlapping_dmrs, np.ndarray):
            overlapping_dmrs = overlapping_dmrs.tolist()
        elif isinstance(overlapping_dmrs, str):
            try:
                overlapping_dmrs = ast.literal_eval(overlapping_dmrs)
            except (ValueError, SyntaxError):
                overlapping_dmrs = []
        elif overlapping_dmrs is None:
            overlapping_dmrs = []
        
        # Skip if no overlapping locations
        if not overlap_locations or len(overlap_locations) == 0:
            continue
        
        # Process each overlapping region
        for i, overlap_region in enumerate(overlap_locations):
            # Handle nested numpy arrays - extract the inner array
            if isinstance(overlap_region, np.ndarray):
                overlap_region = overlap_region.tolist()
            
            if len(overlap_region) >= 2:  # Ensure we have start and end positions
                overlap_start, overlap_end = overlap_region[0], overlap_region[1]
                
                # Extract corresponding sequences for the overlapping region
                overlap_input_ids, overlap_methylation_ids = extract_sequence_for_region(
                    row['input_ids'], 
                    row['methylation_ids'], 
                    row['read_start'], 
                    row['read_end'],
                    overlap_start, 
                    overlap_end
                )
                
                # Get corresponding areastat (if available)
                areastat = areastats[i] if i < len(areastats) else np.nan
                
                # Get corresponding DMR type (if available)
                dmr_type = overlapping_dmrs[i] if i < len(overlapping_dmrs) else 'Unknown'
                
                # Create new record
                new_record = {
                    'read_name': row['read_name'],
                    'input_ids': overlap_input_ids,
                    'methylation_ids': overlap_methylation_ids,
                    'read_start': overlap_start,
                    'read_end': overlap_end,
                    'mapping_quality': row.get('mapping_quality', np.nan),
                    'areastat': areastat,
                    'coverage': row['coverage'],
                    'chromosome': row['chromosome'],
                    'dmr_type': dmr_type,
                    'original_file': row['original_file'],
                    'label': row['label']
                }
                
                new_records.append(new_record)
    
    # Create new DataFrame
    result_df = pd.DataFrame(new_records)
    
    return result_df

def extract_sequence_for_region(input_ids: str, methylation_ids: str, 
                               read_start: int, read_end: int,
                               overlap_start: int, overlap_end: int) -> Tuple[str, str]:
    """
    Extract the portion of input_ids and methylation_ids that corresponds to the overlapping region.
    
    Parameters:
    input_ids (str): DNA sequence
    methylation_ids (str): Methylation status sequence
    read_start (int): Start position of the read
    read_end (int): End position of the read
    overlap_start (int): Start position of the overlap
    overlap_end (int): End position of the overlap
    
    Returns:
    Tuple[str, str]: Extracted input_ids and methylation_ids for the overlapping region
    """
    
    # Calculate the relative positions within the sequences
    read_length = read_end - read_start
    sequence_length = len(input_ids)
    
    # Calculate scaling factor (in case sequence length != read length)
    scale_factor = sequence_length / read_length if read_length > 0 else 1
    
    # Calculate relative start and end positions in the sequence
    rel_overlap_start = max(0, overlap_start - read_start)
    rel_overlap_end = min(read_length, overlap_end - read_start)
    
    # Scale to sequence indices
    seq_start_idx = int(rel_overlap_start * scale_factor)
    seq_end_idx = int(rel_overlap_end * scale_factor)
    
    # Ensure indices are within bounds
    seq_start_idx = max(0, min(seq_start_idx, len(input_ids)))
    seq_end_idx = max(seq_start_idx, min(seq_end_idx, len(input_ids)))
    
    # Extract sequences
    overlap_input_ids = input_ids[seq_start_idx:seq_end_idx]
    overlap_methylation_ids = methylation_ids[seq_start_idx:seq_end_idx] if len(methylation_ids) > seq_start_idx else ''
    
    return overlap_input_ids, overlap_methylation_ids

In [ ]:
# data_prefix = "../Data/Curated/rrms_dmrs_only_mincov40_corrected/chr"
# for chr_n in range(1,23,1):
#     data_path =  data_prefix+str(chr_n)
#     for file in ["train", "valid", "test", "rest"]:
#         chr = data_path.split("/")[-1]
#         new_dir = data_path.replace(f"{chr}", f"{chr}_cpg_counts_selected")
#         new_path = new_dir + f"/{file}.parquet"
#         print(new_path)
#         if not os.path.isdir(new_dir):
#             os.mkdir(new_dir)
#         data = pd.read_parquet(data_path+f"/{file}.parquet")
#         dna, methylation, labels = data["input_ids"],data["methylation_ids"], data["label"]
#         data_optimal = extract_optimal_subsequence_dataframe_chunked(data, target_read_length=1000, selection_criteria="counts")
#         data_optimal.rename(columns={"selected_input_ids": "input_ids", "selected_methylation_ids": "methylation_ids"}, inplace=True)
#         # if not os.path.exists(new_dir):
#         #     os.mkdir(new_dir)
#         # data_optimal = extract_overlapping_regions(data)
#         data_optimal.to_parquet(new_path)

In [6]:
import numpy as np

In [7]:
dismir_instance = Dismir(150,*[os.path.join(data_path,  x) for x in ['train.parquet', 'test.parquet', 'valid.parquet']], flavour="lstm", 
                         num_labels=40, classifier_type="vanilla",
                         num_dmr_labels=40,
                         dmr_label_col="dmr_ctype_label",
                         methylation_column ="pattern",dna_column="seq")

Using vanilla classifier with FC layers


In [ ]:
# train_dir = 'x:\\KULeuven-Masters\\Master Thesis\\methyldl\\methyldl\\Tutorials\\dismir\\minigru_rrms1'

In [8]:
train_dir = '/home/luna.kuleuven.be/u0169940/Repos/methyldl/Tutorials/Dismir_attentionClassifier_DMRs_stratified_hg38_dmr_ctype_label_mincpg_4_minlen_10'

In [9]:
if not os.path.exists(train_dir):
    os.mkdir(train_dir)

In [10]:
import numpy as np

In [11]:
from matplotlib import pyplot as plt

In [12]:
data_path

'/home/luna.kuleuven.be/u0169940/Data/Loyfer/TrainingDataWithRejection_hg38_mincpg_4_minlen_10/'

In [ ]:
# data = pd.read_parquet(data_path+"train.parquet")
# # data_filtered = data.loc[data["sum_abs_areastat"]>4500,].reset_index() #TEMP --> Filter by Area Stat
# dna = data["input_ids"]
# methylation = data["methylation_ids"]
# labels = data["label"]
# [(x,dna[0][x:x+2]) for x in range(150) if dna[0][x:x+2]=="CG"]

In [13]:
dismir_instance.train(epochs=500,batch_size=512*8, train_dir=train_dir, patience=100, optimizer_type="SGD",lr=0.0025, momentum=0.9, variable_length=False, nesterov=False)

Preparing data for fixed-length training...
... Train is ready
... Valid is ready
Start fixed-length training...
Epoch [1/500] Train Loss: 3.6854, Train Acc: 0.1285 | Val Loss: 3.6775, Val Acc: 0.4914
Epoch [2/500] Train Loss: 3.6697, Train Acc: 0.4939 | Val Loss: 3.6609, Val Acc: 0.4914
Epoch [3/500] Train Loss: 3.6526, Train Acc: 0.4939 | Val Loss: 3.6436, Val Acc: 0.4914
Epoch [4/500] Train Loss: 3.6345, Train Acc: 0.4939 | Val Loss: 3.6250, Val Acc: 0.4914
Epoch [5/500] Train Loss: 3.6142, Train Acc: 0.4939 | Val Loss: 3.6032, Val Acc: 0.4914
Epoch [6/500] Train Loss: 3.5891, Train Acc: 0.4939 | Val Loss: 3.5743, Val Acc: 0.4914
Epoch [7/500] Train Loss: 3.5538, Train Acc: 0.4939 | Val Loss: 3.5324, Val Acc: 0.4914
Epoch [8/500] Train Loss: 3.5044, Train Acc: 0.4939 | Val Loss: 3.4805, Val Acc: 0.4914
Epoch [9/500] Train Loss: 3.4542, Train Acc: 0.4939 | Val Loss: 3.4412, Val Acc: 0.4914
Epoch [10/500] Train Loss: 3.4154, Train Acc: 0.4939 | Val Loss: 3.4109, Val Acc: 0.4914
Epoch 

KeyboardInterrupt: 

In [ ]:
# torch.save(dismir_instance.model.state_dict(), os.path.join(train_dir, "weight_final.pt"))

In [ ]:
plt.plot(pd.DataFrame(dismir_instance.history)["train_loss"])
plt.plot(pd.DataFrame(dismir_instance.history)["val_loss"])

In [ ]:
import mlflow

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [ ]:
logged_model = 'runs:/17343621270843cca1d08700d4925f2a/model_chromosome_chr19'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)

In [ ]:
dismir_instance.model = loaded_model.get_raw_model()

In [ ]:
import torch
dismir_instance.model.load_state_dict(torch.load(f"{train_dir}\\weight.pt"))

In [ ]:
file = "test_whole_reads"
data = pd.read_parquet(data_path+f"/{file}.parquet")
# data = data.loc[(data["original_file"] == "COLO829BL_5.bam") | (data["original_file"] =="COLO829_5.bam"),]
dna, methylation, labels = data["input_ids"],data["methylation_ids"], data["label"]

In [ ]:
import numpy as np

In [ ]:
from typing import List, Dict
import pandas as pd
import numpy as np

def split_long_reads(df: pd.DataFrame, max_read_length: int) -> pd.DataFrame:
    """
    Split DNA reads longer than max_read_length into smaller chunks.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing DNA read data
    max_read_length : int
        Maximum allowed read length for splitting
    
    Returns:
    --------
    pd.DataFrame
        Transformed dataset with split reads and calculated CpG counts
    """
    
    def count_cpgs(methylation_string: str) -> int:
        """Count CpG sites (represented by '1' or '2' in methylation_ids)
        0 - unmethylated C, 1 - methylated C, 2 - methylation status unknown"""
        return sum(1 for char in methylation_string if char in ['0', '1'])
    
    def split_single_read(row: pd.Series) -> List[Dict]:
        """Split a single read into chunks if it exceeds max_read_length"""
        # Extract relevant fields directly from the pandas Series
        read_name = row['read_name']
        input_ids = row['input_ids']
        methylation_ids = row['methylation_ids']
        chromosome = row['chromosome']
        original_file = row['original_file']
        label = row['label']
        
        # Get the actual sequence length
        sequence_length = len(input_ids)
        
        # If the read is within the max length, return as is
        if sequence_length <= max_read_length:
            return [{
                'read_name': read_name,
                'input_ids': input_ids,
                'methylation_ids': methylation_ids,
                'chromosome': chromosome,
                'original_file': original_file,
                'label': label,
                'num_cpgs': count_cpgs(methylation_ids)
            }]
        
        # Split the read into chunks
        chunks = []
        for i in range(0, sequence_length, max_read_length):
            end_idx = min(i + max_read_length, sequence_length)
            
            # Extract the chunk
            chunk_input_ids = input_ids[i:end_idx]
            chunk_methylation_ids = methylation_ids[i:end_idx]
            
            chunks.append({
                'read_name': read_name,
                'input_ids': chunk_input_ids,
                'methylation_ids': chunk_methylation_ids,
                'chromosome': chromosome,
                'original_file': original_file,
                'label': label,
                'num_cpgs': count_cpgs(chunk_methylation_ids)
            })
        
        return chunks
    
    # Process all rows
    all_chunks = []
    for _, row in df.iterrows():
        chunks = split_single_read(row)
        all_chunks.extend(chunks)
    
    # Convert to DataFrame
    result_df = pd.DataFrame(all_chunks)
    
    return result_df

In [ ]:
data_chunked = split_long_reads(data,1000)

In [ ]:
dna, methylation, labels = data_chunked["input_ids"],data_chunked["methylation_ids"], data_chunked["label"]

In [ ]:
predictions_proba,predictions = dismir_instance.predict(dna,methylation_sequences=methylation,batch_size=500, parallel_scan=False)

In [ ]:
data_chunked["predictions_proba"] = predictions_proba
data_chunked["predictions_weighted"] = data_chunked["predictions_proba"] * data_chunked["num_cpgs"]
data_chunked_agg = data_chunked.groupby("read_name").agg(
    predictions_weighted=pd.NamedAgg(column="predictions_weighted", aggfunc="sum"),
    num_cpgs=pd.NamedAgg(column="num_cpgs", aggfunc="sum"),
    label=pd.NamedAgg(column="label", aggfunc="min"),
    )
data_chunked_agg["num_cpgs"] = [max(x,1) for x in data_chunked_agg["num_cpgs"]]

data_chunked_agg["predictions_weighted"] = data_chunked_agg["predictions_weighted"]/data_chunked_agg["num_cpgs"]
data_chunked_agg["predictions"] = data_chunked_agg["predictions_weighted"]>0.5
data_chunked_agg = data_chunked_agg.loc[data_chunked_agg["num_cpgs"]>0,]
labels, predictions, predictions_proba = data_chunked_agg["label"], data_chunked_agg["predictions"], data_chunked_agg["predictions_weighted"]

In [ ]:
misclassified_reads = data_chunked_agg.loc[data_chunked_agg["label"]!=data_chunked_agg["predictions"]].reset_index()["read_name"].to_numpy()

In [ ]:
data["missclassified"] = data["read_name"].apply(lambda x: x in misclassified_reads)

In [ ]:
# data.groupby("dmr_name").agg(
#     percentage_missclassified = pd.NamedAgg(column="missclassified",aggfunc="mean")
# ).sort_values("percentage_missclassified")

In [ ]:
# target_indexes = list(data.loc[data["read_length"]<=1000].index)
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, auc, roc_curve,roc_auc_score
import numpy as np

# Set Matplotlib styling
plt.rcParams.update({
    "font.size": 12,
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "grid.color": "lightgray",
    "grid.linestyle": "-",
})


labels_filtered = labels
predictions_filtered = predictions
predictions_proba_filtered = predictions_proba
# labels_filtered = labels_filtered[target_indexes]
# predictions_filtered = predictions_filtered[target_indexes]
# predictions_proba_filtered = predictions_proba_filtered[target_indexes]
# Compute confusion matrix
# Compute ROC and AUC
fpr, tpr, thresholds = roc_curve(labels_filtered, predictions_proba_filtered)
auc_val = auc(fpr, tpr)
best_treshold = thresholds[np.argmax(tpr-fpr)]
predictions_filtered = predictions_proba_filtered > best_treshold

cf_matrix = confusion_matrix(labels_filtered, predictions_filtered)

# Create the heatmap using Matplotlib
fig, ax = plt.subplots(1, figsize=(6, 6))
cax = ax.matshow(cf_matrix, cmap="PiYG")
plt.colorbar(cax)

# Annotate the heatmap
for (i, j), val in np.ndenumerate(cf_matrix):
    ax.text(j, i, f"{val}", ha="center", va="center", color="white" if val > cf_matrix.max() / 2 else "black")

# Set axis labels and tick marks
ax.set_xlabel("Prediction", fontsize=12)
ax.set_ylabel("Ground-truth", fontsize=12)
ax.set_xticks([0, 1])
ax.set_xticklabels(["N", "T"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["N", "T"])

# Set title with AUC value
ax.set_title(f"DISMIR Original (ROC-AUC: {auc_val:.3f})", fontsize=14)

# Show plot
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def plot_filled_kde(values, groups,alpha=0.4, ax=None,bw_adjust=0.3):
    """
    Plots overlapping KDE (Kernel Density Estimation) curves with filled areas for two groups (0 and 1).
    Also computes Cohen's d to measure effect size.

    Parameters:
    - values: A list or numpy array of numerical values.
    - groups: A list or numpy array of group labels (0 or 1).
    - system_index: Index or name of the system being analyzed.
    - alpha: Transparency level for the filled area under KDE curves (default is 0.4).
    - ax: Matplotlib axis to plot on (optional).
    """
    # Convert to numpy arrays
    values = np.array(values)
    groups = np.array(groups)

    # Ensure groups contain only 0s and 1s
    unique_groups = np.unique(groups)
    if len(unique_groups) > 2 or set(unique_groups) - {0, 1}:
        raise ValueError("Groups should contain only 0 and 1.")

    # Separate values by group
    values_0 = values[groups == 0]  # Impostor scores
    values_1 = values[groups == 1]  # Genuine scores


    # Compute Uniquness and permanence
    uniqueness = np.abs(np.mean(values_0) - np.mean(values_1))

    n1, n2 = len(values_0), len(values_1)
    std1, std2 = np.std(values_0, ddof=1), np.std(values_1, ddof=1)  # Unbiased std
    pooled_std = np.sqrt( ((n1-1)*std1**2 + (n2-1)*std2**2)/(n1+n2-2))
    #permanence = pooled_std

    # If no axis is provided, create one
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    # KDE plots with filled area
    sns.kdeplot(values_0, color="blue", fill=True, alpha=alpha, label="Normal", linewidth=2, ax=ax,bw_adjust=bw_adjust)
    sns.kdeplot(values_1, color="red", fill=True, alpha=alpha, label="Tumor", linewidth=2, ax=ax,bw_adjust=bw_adjust)

    # Labels and legend
    ax.set_xlabel("Score")
    ax.set_ylabel("Density")
    ax.set_title(f"Read level predictions")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.6)

    # If ax is None, show the plot
    if ax is None:
        plt.show()


In [ ]:
plot_filled_kde(predictions, labels)

In [6]:
from methyldl.modelling.evaluation import calculate_metric_with_sklearn
import numpy as np
import torch

In [5]:
import torch.nn as nn

In [70]:
probs = np.array([[0.48395887, 0.494468  ]])